In [1]:
import os
import torch
import torch
import os
from collections import defaultdict
print(os.getcwd())
# os.chdir('/home/seunghan.lee/workspace/ts-rag/old_checkpoints/checkpoints')
os.chdir('/home/seunghan.lee/workspace/ts-rag/checkpoints')

/home/seunghan.lee/workspace/experiments


In [2]:
#[x for x in os.listdir() if 'R_as' in x]

In [3]:
# fn1 = 'pretrain_Chronos_lb512_X-euclidean_k3_TabPFN_dualhead_learnable'
# fn2 = 'pretrain_Chronos_lb512_X-euclidean_k1_TabPFN_dualhead_learnable'
fn3 = 'pretrain_Chronos_lb512_X-cosine-norm_k3_TabPFN_dualhead_R_as_key_l0.7'

# fn1 = 'pretrain_Chronos_lb512_X-cosine-norm_k1_TabPFN_dualhead_l0.3'
# fn2 = 'pretrain_Chronos_lb512_X-cosine-norm_k3_TabPFN_dualhead_l0.4'
# fn3 = 'pretrain_Chronos_lb512_X-cosine-norm_k1_TabPFN_dualhead_l0.5'

In [4]:
# x1 = torch.load(os.path.join(fn1,'best.pth'),map_location='cpu')
# x2 = torch.load(os.path.join(fn2,'best.pth'),map_location='cpu')
x3 = torch.load(os.path.join(fn3,'best.pth'),map_location='cpu')

In [5]:
ckpt = torch.load(os.path.join('base', 'autogluon_model.pth'), map_location='cpu')
#ckpt = torch.load(os.path.join('chronos-bolt', 'best.pth'), map_location='cpu')
#ckpt = torch.load(os.path.join(fn3, 'best.pth'), map_location='cpu')
state_dict = ckpt.get('state_dict', ckpt)
param_cnt = defaultdict(int)
for k, v in state_dict.items():
    top_key = k.split('.', 1)[0]
    if 'gate' not in top_key:
        param_cnt[top_key] += v.numel()

total = sum(param_cnt.values())
for k in sorted(param_cnt):
    print(f"{k:30s} {param_cnt[k]:>12d} ({param_cnt[k]/total:.2%})")

print(f"\nTotal parameters: {total}")

decoder                           113276544 (55.18%)
encoder                            84955776 (41.38%)
input_patch_embedding               2486784 (1.21%)
output_patch_embedding              4575360 (2.23%)
shared                                 1536 (0.00%)

Total parameters: 205296000


In [6]:
# ckpt = torch.load(os.path.join('base', 'autogluon_model.pth'), map_location='cpu')
ckpt = torch.load(os.path.join('chronos-bolt', 'best.pth'), map_location='cpu')
#ckpt = torch.load(os.path.join(fn3, 'best.pth'), map_location='cpu')
state_dict = ckpt.get('state_dict', ckpt)
param_cnt = defaultdict(int)
for k, v in state_dict.items():
    top_key = k.split('.', 1)[0]
    if 'gate' not in top_key:
        param_cnt[top_key] += v.numel()

total = sum(param_cnt.values())
for k in sorted(param_cnt):
    print(f"{k:30s} {param_cnt[k]:>12d} ({param_cnt[k]/total:.2%})")

print(f"\nTotal parameters: {total}")

decoder                           113276544 (54.08%)
encode_mlp                           640512 (0.31%)
encoder                            84955776 (40.56%)
ffn                                 1181184 (0.56%)
input_patch_embedding               2486784 (1.19%)
mha                                 2362368 (1.13%)
output_patch_embedding              4575360 (2.18%)
shared                                 1536 (0.00%)

Total parameters: 209480064


In [7]:
#encoder_mlp/ffn/mha
#(640512+1181184+2362368)/209480064
#2.00%

In [8]:
# ckpt = torch.load(os.path.join('base', 'autogluon_model.pth'), map_location='cpu')
# ckpt = torch.load(os.path.join('chronos-bolt', 'best.pth'), map_location='cpu')
ckpt = torch.load(os.path.join(fn3, 'best.pth'), map_location='cpu')
state_dict = ckpt.get('state_dict', ckpt)
param_cnt = defaultdict(int)
for k, v in state_dict.items():
    top_key = k.split('.', 1)[0]
    if 'gate' not in top_key:
        param_cnt[top_key] += v.numel()

total = sum(param_cnt.values())
for k in sorted(param_cnt):
    print(f"{k:30s} {param_cnt[k]:>12d} ({param_cnt[k]/total:.2%})")

print(f"\nTotal parameters: {total}")

cross_mha                           2362368 (1.10%)
decoder                           113276544 (52.93%)
encode_mlp_x                         984576 (0.46%)
encode_mlp_y                         640512 (0.30%)
encoder                            84955776 (39.70%)
ffn_cross                           1181184 (0.55%)
ffn_self                            1181184 (0.55%)
input_patch_embedding               2486784 (1.16%)
output_patch_embedding              4575360 (2.14%)
self_mha                            2362368 (1.10%)
shared                                 1536 (0.00%)

Total parameters: 214008192


In [19]:
#encoder_mlp/ffn/mha
(2362368+984576+640512+1181184+1181184+2362368)/214008192
#0.30%

0.04070961919065229

In [142]:
import torch
import os

ckpt = torch.load(os.path.join('base', 'autogluon_model.pth'), map_location='cpu')

# 보통 checkpoint에 state_dict가 있거나, 바로 state_dict인 경우가 있음
state_dict = ckpt.get('state_dict', ckpt)

encode_cnt = 0
non_encode_cnt = 0
cross_mha = 0
self_mha = 0



for k, v in state_dict.items():
    num_params = v.numel()
    if "encode_mlp." in k:
        encode_cnt += num_params
    else:
        non_encode_cnt += num_params
        if 'cross_mha.' in k:
            cross_mha += num_params
        if 'self_mha.' in k:
            self_mha += num_params
            

total_cnt = encode_cnt + non_encode_cnt

print(f'encode_mlp params: {encode_cnt}')
print(f'non-encode_mlp params: {non_encode_cnt}')
print(f'cross_mha params: {cross_mha}')
print(f'self_mha params: {self_mha}')
print(f'ratio (encode / total): {encode_cnt / total_cnt:.4f}')
print(f'ratio (non-encode / total): {non_encode_cnt / total_cnt:.4f}')
b = list(state_dict.keys())

encode_mlp params: 0
non-encode_mlp params: 205296000
cross_mha params: 0
self_mha params: 0
ratio (encode / total): 0.0000
ratio (non-encode / total): 1.0000


In [126]:
import torch
import os

ckpt = torch.load(os.path.join(fn3, 'best.pth'), map_location='cpu')

# 보통 checkpoint에 state_dict가 있거나, 바로 state_dict인 경우가 있음
state_dict = ckpt.get('state_dict', ckpt)

encode_cnt = 0
non_encode_cnt = 0
cross_mha = 0
self_mha = 0

for k, v in state_dict.items():
    num_params = v.numel()
    if "encode_mlp." in k:
        encode_cnt += num_params
    else:
        non_encode_cnt += num_params
        if 'cross_mha.' in k:
            cross_mha += num_params
        if 'self_mha.' in k:
            self_mha += num_params
            

total_cnt = encode_cnt + non_encode_cnt

print(f'encode_mlp params: {encode_cnt}')
print(f'non-encode_mlp params: {non_encode_cnt}')
print(f'cross_mha params: {cross_mha}')
print(f'self_mha params: {self_mha}')
print(f'ratio (encode / total): {encode_cnt / total_cnt:.4f}')
print(f'ratio (non-encode / total): {non_encode_cnt / total_cnt:.4f}')
c = list(state_dict.keys())

encode_mlp params: 640512
non-encode_mlp params: 213564289
cross_mha params: 2362368
self_mha params: 2362368
ratio (encode / total): 0.0030
ratio (non-encode / total): 0.9970


In [136]:
print('a-b',set([x.split('.')[0] for x in list(set(a)-set(b))]))
print('c-b',set([x.split('.')[0] for x in list(set(c)-set(b))]))


a-b {'ffn', 'encode_mlp', 'mha', 'gate_layer'}
c-b {'ffn_self', 'mix_gate', 'self_mha', 'cross_mha', 'ffn_cross', 'encode_mlp'}


In [128]:
# 'output_patch_embedding.hidden_layer.weight',
#  'output_patch_embedding.hidden_layer.bias',
#  'output_patch_embedding.output_layer.weight',
#  'output_patch_embedding.output_layer.bias',
#  'output_patch_embedding.residual_layer.weight',
#  'output_patch_embedding.residual_layer.bias',
#  'encode_mlp.0.weight',
#  'encode_mlp.0.bias',
#  'encode_mlp.2.weight',
#  'encode_mlp.2.bias',
#  'cross_mha.in_proj_weight',
#  'cross_mha.in_proj_bias',
#  'cross_mha.out_proj.weight',
#  'cross_mha.out_proj.bias',
#  'self_mha.in_proj_weight',
#  'self_mha.in_proj_bias',
#  'self_mha.out_proj.weight',
#  'self_mha.out_proj.bias',
#  'ffn_cross.0.weight',
#  'ffn_cross.0.bias',
#  'ffn_cross.2.weight',
#  'ffn_cross.2.bias',
#  'ffn_self.0.weight',
#  'ffn_self.0.bias',
#  'ffn_self.2.weight',
#  'ffn_self.2.bias',
#  'mix_gate.0.weight',
#  'mix_gate.0.bias',
#  'mix_gate.2.weight',
# #  'mix_gate.2.bias']

In [83]:
print(x1['self_mha.in_proj_weight'])
print(x2['self_mha.in_proj_weight'])
print(x3['self_mha.in_proj_weight'])

tensor([[-0.0118, -0.0262, -0.0017,  ...,  0.0186, -0.0373,  0.0034],
        [-0.0128, -0.0305,  0.0353,  ...,  0.0111,  0.0203,  0.0288],
        [-0.0414, -0.0344,  0.0032,  ...,  0.0267, -0.0170, -0.0012],
        ...,
        [-0.0433, -0.0209,  0.0063,  ...,  0.0373,  0.0367,  0.0191],
        [ 0.0439,  0.0256,  0.0340,  ...,  0.0064, -0.0103,  0.0382],
        [ 0.0006, -0.0329, -0.0394,  ..., -0.0314, -0.0189, -0.0363]])
tensor([[-0.0118, -0.0262, -0.0017,  ...,  0.0186, -0.0373,  0.0034],
        [-0.0128, -0.0305,  0.0353,  ...,  0.0111,  0.0203,  0.0288],
        [-0.0414, -0.0344,  0.0032,  ...,  0.0267, -0.0170, -0.0012],
        ...,
        [-0.0433, -0.0209,  0.0063,  ...,  0.0373,  0.0367,  0.0191],
        [ 0.0439,  0.0256,  0.0340,  ...,  0.0064, -0.0103,  0.0382],
        [ 0.0006, -0.0329, -0.0394,  ..., -0.0314, -0.0189, -0.0363]])
tensor([[-0.0118, -0.0262, -0.0017,  ...,  0.0186, -0.0373,  0.0034],
        [-0.0128, -0.0305,  0.0353,  ...,  0.0111,  0.0203,  0

In [32]:
x2['cross_mha.in_proj_weight']

tensor([[-0.0324,  0.0012,  0.0213,  ...,  0.0368,  0.0217,  0.0206],
        [ 0.0241,  0.0138, -0.0186,  ...,  0.0326, -0.0369,  0.0334],
        [-0.0117,  0.0174, -0.0279,  ...,  0.0331, -0.0016, -0.0054],
        ...,
        [ 0.0081,  0.0154,  0.0333,  ..., -0.0392, -0.0157,  0.0405],
        [-0.0047, -0.0052, -0.0175,  ..., -0.0220, -0.0007, -0.0349],
        [ 0.0180, -0.0165,  0.0136,  ..., -0.0287,  0.0297, -0.0053]])

In [15]:
x1['self_mha.out_proj.weight']

tensor([[0.0000e+00, 0.0000e+00, 1.8816e-40,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00]])

In [ ]:
x